In [2]:
import pandas as pd
import plotly.express as px 
import plotly.graph_objects as go
import numpy as np


In [3]:
raw_data = pd.read_excel('get_around_delay_analysis.xlsx')
raw_data.shape

(21310, 7)

In [4]:
raw_data.describe(include='all')

,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
count,21310.000000,21310.000000,21310,21310,16346.000000,1841.000000,1841.000000
unique,NaN,NaN,2,2,NaN,NaN,NaN
top,NaN,NaN,mobile,ended,NaN,NaN,NaN
freq,NaN,NaN,17003,18045,NaN,NaN,NaN
mean,549712.880338,350030.603426,NaN,NaN,59.701517,550127.411733,279.288430
std,13863.446964,58206.249765,NaN,NaN,1002.561635,13184.023111,254.594486
min,504806.000000,159250.000000,NaN,NaN,-22433.000000,505628.000000,0.000000
25%,540613.250000,317639.000000,NaN,NaN,-36.000000,540896.000000,60.000000
50%,550350.000000,368717.000000,NaN,NaN,9.000000,550567.000000,180.000000
75%,560468.500000,394928.000000,NaN,NaN,67.000000,560823.000000,540.000000


In [5]:
raw_data.head()

,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
0,505000,363965,mobile,canceled,NaN,NaN,NaN
1,507750,269550,mobile,ended,-81.0,NaN,NaN
2,508131,359049,connect,ended,70.0,NaN,NaN
3,508865,299063,connect,canceled,NaN,NaN,NaN
4,511440,313932,mobile,ended,NaN,NaN,NaN


In [6]:
temp_data = raw_data[raw_data['state'] == 'ended'] #not the canceled ones
temp_data = temp_data.merge(temp_data[['rental_id', 'delay_at_checkout_in_minutes', 'checkin_type']], 
                            how='left', left_on='previous_ended_rental_id', right_on='rental_id',
                            suffixes=['','_previous']) #add the Chackout delay of the previous rental
temp_data = temp_data.drop(['rental_id_previous', 'state'], axis=1) #drop duplicate
temp_data['type_issame'] = temp_data['checkin_type'] == temp_data['checkin_type_previous'] 
data = temp_data[temp_data['delay_at_checkout_in_minutes_previous'].notna()]
data['overlap'] = data['delay_at_checkout_in_minutes_previous'] - data['time_delta_with_previous_rental_in_minutes'] #if overlap negative, then there is enough delta
data['previous_late'] = data['delay_at_checkout_in_minutes_previous']>0
data['impacted'] = data['overlap'] > 0
data.head()

C:\Users\dubos\AppData\Local\Temp\ipykernel_3868\2928866813.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['overlap'] = data['delay_at_checkout_in_minutes_previous'] - data['time_delta_with_previous_rental_in_minutes'] #if overlap negative, then there is enough delta
C:\Users\dubos\AppData\Local\Temp\ipykernel_3868\2928866813.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['previous_late'] = data['delay_at_checkout_in_minutes_previous']>0
C:\Users\dubos\AppData\Local\Temp\ipykernel_3868\2

,rental_id,car_id,checkin_type,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes,delay_at_checkout_in_minutes_previous,checkin_type_previous,type_issame,overlap,previous_late,impacted
4,511639,370585,connect,-15.0,563782.0,570.0,136.0,connect,True,-434.0,True,False
13,519491,312389,mobile,58.0,545639.0,420.0,140.0,mobile,True,-280.0,True,False
24,525044,349751,mobile,NaN,510607.0,60.0,-113.0,mobile,True,-173.0,False,False
30,528808,181625,connect,-76.0,557404.0,330.0,-352.0,connect,True,-682.0,False,False
50,533670,320824,connect,-6.0,556563.0,630.0,23.0,connect,True,-607.0,True,False


In [7]:
data.describe(include='all')

,rental_id,car_id,checkin_type,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes,delay_at_checkout_in_minutes_previous,checkin_type_previous,type_issame,overlap,previous_late,impacted
count,1523.000000,1523.000000,1523,1476.000000,1523.000000,1523.000000,1523.000000,1523,1523,1523.000000,1523,1523
unique,NaN,NaN,2,NaN,NaN,NaN,NaN,2,2,NaN,2,2
top,NaN,NaN,mobile,NaN,NaN,NaN,NaN,mobile,True,NaN,True,False
freq,NaN,NaN,853,NaN,NaN,NaN,NaN,859,1509,NaN,767,1342
mean,552227.014445,351745.271832,NaN,28.134146,549869.814839,273.683519,-22.427446,NaN,NaN,-296.110965,NaN,NaN
std,12956.650124,54244.297467,NaN,429.744537,13383.286259,254.809242,443.314397,NaN,NaN,505.846041,NaN,NaN
min,505560.000000,159533.000000,NaN,-1707.000000,505628.000000,0.000000,-4624.000000,NaN,NaN,-4684.000000,NaN,NaN
25%,543567.500000,323987.000000,NaN,-47.000000,540738.000000,60.000000,-52.500000,NaN,NaN,-572.500000,NaN,NaN
50%,552310.000000,368432.000000,NaN,4.000000,550317.000000,180.000000,1.000000,NaN,NaN,-199.000000,NaN,NaN
75%,562481.000000,391449.000000,NaN,54.000000,560749.500000,540.000000,41.000000,NaN,NaN,-47.000000,NaN,NaN


In [8]:
data.to_csv('data_clean.csv')

In [9]:
data_late = data.groupby('time_delta_with_previous_rental_in_minutes').agg({'previous_late': ['count', 'sum', 'mean'], 'impacted' : ['sum', 'mean']})
                                                                           
# df.groupby('A').agg({'B': ['min', 'max'], 'C': 'sum'})
data_late['previous_late', 'countcumsum'] = data_late['previous_late', 'count'].cumsum()
data_late['previous_late', 'cumsum'] = data_late['previous_late', 'sum'].cumsum()
data_late['impacted', 'cumsum'] = data_late['impacted', 'sum'].cumsum()
data_late['impacted_by_lateness'] = data_late['impacted', 'mean']/ data_late['previous_late', 'mean']

data_late


previous_late                \
                                                   count sum      mean   
time_delta_with_previous_rental_in_minutes                               
0.0                                                  232  99  0.426724   
30.0                                                 112  57  0.508929   
60.0                                                 148  64  0.432432   
90.0                                                  73  46  0.630137   
120.0                                                122  72  0.590164   
150.0                                                 55  29  0.527273   
180.0                                                 71  38  0.535211   
210.0                                                 34  18  0.529412   
240.0                                                 56  28  0.500000   
270.0                                                 32  22  0.687500   
300.0                                                 32  21  0.656250   
330.0                                                 21   7  0.333333   
360.0                                                 34  26  0.764706   
390.0                                                 11  10  0.909091   
420.0                                                 29  17  0.586207   
450.0                                                 16   9  0.562500   
480.0                                                 31  14  0.451613   
510.0                                                 28  12  0.428571   
540.0                                                 32  19  0.593750   
570.0                                                 35  16  0.457143   
600.0                                                 54  25  0.462963   
630.0                                                 42  23  0.547619   
660.0                                                 61  23  0.377049   
690.0                                                 49  23  0.469388   
720.0                                                113  49  0.433628   

                                           impacted           previous_late  \
                                                sum      mean   countcumsum   
time_delta_with_previous_rental_in_minutes                                    
0.0                                              99  0.426724           232   
30.0                                             28  0.250000           344   
60.0                                             21  0.141892           492   
90.0                                              7  0.095890           565   
120.0                                            10  0.081967           687   
150.0                                             2  0.036364           742   
180.0                                             3  0.042254           813   
210.0                                             1  0.029412           847   
240.0                                             3  0.053571           903   
270.0                                             0  0.000000           935   
300.0                                             1  0.031250           967   
330.0                                             0  0.000000           988   
360.0                                             0  0.000000          1022   
390.0                                             0  0.000000          1033   
420.0                                             1  0.034483          1062   
450.0                                             1  0.062500          1078   
480.0                                             0  0.000000          1109   
510.0                                             1  0.035714          1137   
540.0                                             1  0.031250          1169   
570.0                                             1  0.028571          1204   
600.0                                             0  0.000000          1258   
630.0                                             0  0.000000          130

In [10]:
# px.scatter(data)

In [11]:
dataset = pd.read_csv("get_around_pricing_project.csv", index_col=0)
dataset.head()

,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
1,Citroën,13929,317,petrol,grey,convertible,True,True,False,False,False,True,True,264
2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183


In [12]:
dataset.describe(include='all')

,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
count,4843,4.843000e+03,4843.00000,4843,4843,4843,4843,4843,4843,4843,4843,4843,4843,4843.000000
unique,28,NaN,NaN,4,10,8,2,2,2,2,2,2,2,NaN
top,Citroën,NaN,NaN,diesel,black,estate,True,True,False,False,False,False,True,NaN
freq,969,NaN,NaN,4641,1633,1606,2662,3839,3865,3881,2613,3674,4514,NaN
mean,NaN,1.409628e+05,128.98823,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,121.214536
std,NaN,6.019674e+04,38.99336,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.568268
min,NaN,-6.400000e+01,0.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.000000
25%,NaN,1.029135e+05,100.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,104.000000
50%,NaN,1.410800e+05,120.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,119.000000
75%,NaN,1.751955e+05,135.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,136.000000


In [13]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4843 entries, 0 to 4842
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   model_key                  4843 non-null   object
 1   mileage                    4843 non-null   int64 
 2   engine_power               4843 non-null   int64 
 3   fuel                       4843 non-null   object
 4   paint_color                4843 non-null   object
 5   car_type                   4843 non-null   object
 6   private_parking_available  4843 non-null   bool  
 7   has_gps                    4843 non-null   bool  
 8   has_air_conditioning       4843 non-null   bool  
 9   automatic_car              4843 non-null   bool  
 10  has_getaround_connect      4843 non-null   bool  
 11  has_speed_regulator        4843 non-null   bool  
 12  winter_tires               4843 non-null   bool  
 13  rental_price_per_day       4843 non-null   int64 
dtypes: bool(7), i

In [14]:
obj_col = dataset.select_dtypes(include="object").columns
obj_col
for col in obj_col:
    value_set = list(set(dataset[col]))
    trad_dic = {}
    for i in range(len(value_set)):
        trad_dic[value_set[i]] = i

    dataset[col]  = dataset[col].map(trad_dic)

print(dataset)
    

      model_key  mileage  engine_power  fuel  paint_color  car_type  \
0            24   140411           100     0            4         1   
1            24    13929           317     1            5         1   
2            24   183297           120     0            3         1   
3            24   128035           135     0            7         1   
4            24    97097           160     0            0         1   
...         ...      ...           ...   ...          ...       ...   
4838         21    39743           110     0            4         3   
4839         21    49832           100     0            5         3   
4840         21    19633           110     0            5         3   
4841         21    27920           110     0            2         3   
4842          1   195840           160     0            5         3   

      private_parking_available  has_gps  has_air_conditioning  automatic_car  \
0                          True     True                 False    

In [15]:
# Source - https://stackoverflow.com/a/48271124
# Posted by MaxU - stand with Ukraine, modified by community. See post 'Timeline' for change history
# Retrieved 2026-02-10, License - CC BY-SA 3.0

dataset.loc[:, dataset.dtypes.eq('bool')] = dataset.loc[:, dataset.dtypes.eq('bool')].astype(np.int64)
dataset


C:\Users\dubos\AppData\Local\Temp\ipykernel_3868\1767244680.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1 1 0 ... 0 1 1]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  dataset.loc[:, dataset.dtypes.eq('bool')] = dataset.loc[:, dataset.dtypes.eq('bool')].astype(np.int64)
C:\Users\dubos\AppData\Local\Temp\ipykernel_3868\1767244680.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1 1 0 ... 1 1 1]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  dataset.loc[:, dataset.dtypes.eq('bool')] = dataset.loc[:, dataset.dtypes.eq('bool')].astype(np.int64)
C:\Users\dubos\AppData\Local\Temp\ipykernel_3868\1767244680.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' has dtype

,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,24,140411,100,0,4,1,1,1,0,0,1,1,1,106
1,24,13929,317,1,5,1,1,1,0,0,0,1,1,264
2,24,183297,120,0,3,1,0,0,0,0,1,0,1,101
3,24,128035,135,0,7,1,1,1,0,0,1,1,1,158
4,24,97097,160,0,0,1,1,1,0,0,0,1,1,183
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4838,21,39743,110,0,4,3,0,1,0,0,0,0,1,121
4839,21,49832,100,0,5,3,0,1,0,0,0,0,1,132
4840,21,19633,110,0,5,3,0,1,0,0,0,0,1,130
4841,21,27920,110,0,2,3,1,1,0,0,0,0,1,151


In [16]:
bool_col = dataset.select_dtypes(include="bool").columns
bool_col
for col in bool_col:
    value_set = list(set(dataset[col]))
    trad_dic = {}
    for i in range(len(value_set)):
        trad_dic[value_set[i]] = i

    dataset[col]  = dataset[col].map(trad_dic)

print(dataset)

      model_key  mileage  engine_power  fuel  paint_color  car_type  \
0            24   140411           100     0            4         1   
1            24    13929           317     1            5         1   
2            24   183297           120     0            3         1   
3            24   128035           135     0            7         1   
4            24    97097           160     0            0         1   
...         ...      ...           ...   ...          ...       ...   
4838         21    39743           110     0            4         3   
4839         21    49832           100     0            5         3   
4840         21    19633           110     0            5         3   
4841         21    27920           110     0            2         3   
4842          1   195840           160     0            5         3   

      private_parking_available  has_gps  has_air_conditioning  automatic_car  \
0                             1        1                     0    

In [17]:
brands = list(set(dataset["model_key"]))
dic_brand = {}
for i in range(len(brands)):
    dic_brand[brands[i]] = i
    

dic_brand

{0: 0,
 1: 1,
 2: 2,
 3: 3,
 4: 4,
 5: 5,
 6: 6,
 7: 7,
 8: 8,
 9: 9,
 10: 10,
 11: 11,
 12: 12,
 13: 13,
 14: 14,
 15: 15,
 16: 16,
 17: 17,
 18: 18,
 19: 19,
 20: 20,
 21: 21,
 22: 22,
 23: 23,
 24: 24,
 25: 25,
 26: 26,
 27: 27}

In [18]:
price_car = {
    "Alfa Romeo":'',
    "Audi":44777,
    "BMW":50259,
    "Citroën":22343,
    "Ferrari":'',
    "Fiat":17464,
    "Ford":27032,
    "Honda":''
}

In [19]:
go.Layout?

Init signature:
go.Layout(
    arg=None,
    activeselection=None,
    activeshape=None,
    annotations=None,
    annotationdefaults=None,
    autosize=None,
    autotypenumbers=None,
    barcornerradius=None,
    bargap=None,
    bargroupgap=None,
    barmode=None,
    barnorm=None,
    boxgap=None,
    boxgroupgap=None,
    boxmode=None,
    calendar=None,
    clickmode=None,
    coloraxis=None,
    colorscale=None,
    colorway=None,
    computed=None,
    datarevision=None,
    dragmode=None,
    editrevision=None,
    extendfunnelareacolors=None,
    extendiciclecolors=None,
    extendpiecolors=None,
    extendsunburstcolors=None,
    extendtreemapcolors=None,
    font=None,
    funnelareacolorway=None,
    funnelgap=None,
    funnelgroupgap=None,
    funnelmode=None,
    geo=None,
    grid=None,
    height=None,
    hiddenlabels=None,
    hiddenlabelssrc=None,
    hidesources=None,
    hoverdistance=None,
    hoverlabel=None,
    hovermode=None,
    hoversubplots=None,
    icicl

In [20]:
pd.DataFrame([[7.0, 0.27, 0.36, 20.7, 0.045, 45.0, 170.0, 1.001, 3.0, 0.45, 8.8], [7.0, 0.27, 0.36, 20.7, 0.045, 45.0, 170.0, 1.001, 3.0, 0.45, 8.8]])


,0,1,2,3,4,5,6,7,8,9,10
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.001,3.0,0.45,8.8
1,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.001,3.0,0.45,8.8


In [22]:
input = {
  "input": [[7.0, 0.27, 0.36, 20.7, 0.045, 45.0, 170.0, 1.001, 3.0, 0.45, 8.8], [7.0, 0.27, 0.36, 20.7, 0.045, 45.0, 170.0, 1.001, 3.0, 0.45, 8.8]]
}
pd.DataFrame(input["input"])

,0,1,2,3,4,5,6,7,8,9,10
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.001,3.0,0.45,8.8
1,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.001,3.0,0.45,8.8
